# Respostas às perguntas de negócio (pandas)
Notebook que lê as tabelas `listings`, `sellers` e `products` do PostgreSQL e responde às 5 perguntas do case com **pandas**.

Pré-requisitos: `pip install pandas sqlalchemy psycopg2-binary python-dotenv` e o `.env` com as variáveis `DB_*`.

## 1. Conexão e carga dos dados

In [1]:
import os
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv(find_dotenv(usecwd=True))   

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

JOB_RUN = '2026-10-12 14:28:43.000'

engine = create_engine(URL.create(
    'postgresql+psycopg2',
    username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    database=os.environ['DB_NAME'],
))

In [2]:
with engine.connect() as conn:
    params = {'job_run': JOB_RUN}
    listings = pd.read_sql(text('SELECT * FROM listings WHERE job_run = :job_run'), conn, params=params)
    sellers  = pd.read_sql(text('SELECT * FROM sellers  WHERE job_run = :job_run'), conn, params=params)
    products = pd.read_sql(text('SELECT * FROM products WHERE job_run = :job_run'), conn, params=params)
    etl_runs = pd.read_sql(text('SELECT * FROM etl_runs ORDER BY job_run DESC LIMIT 5'), conn)

for col in ['price', 'price_converted', 'total_sales']:
    listings[col] = pd.to_numeric(listings[col], errors='coerce')
sellers['transactions_total'] = pd.to_numeric(sellers['transactions_total'], errors='coerce')

print(f'JOB_RUN analisado: {JOB_RUN}')
print(f'listings: {len(listings)} | sellers: {len(sellers)} | products: {len(products)}')

JOB_RUN analisado: 2026-10-12 14:28:43.000
listings: 158 | sellers: 65 | products: 31


### Histórico das últimas execuções (`etl_runs`)

In [3]:
etl_runs.head(5)

,job_run,search_term,status,execution_seconds,status_sellers,status_products,status_listings,log_path,error_message
0,2026-10-13 09:27:33,S25 FE,SUCCESS,7.95,LOADED,LOADED,LOADED,logs/etl_2026-10-13_09-27-33.log,None
1,2026-10-13 09:19:39,S25 FE,SUCCESS,7.64,LOADED,LOADED,LOADED,logs/etl_2026-10-13_09-19-39.log,None
2,2026-10-12 14:28:43,S25 FE,SUCCESS,8.90,LOADED,LOADED,LOADED,logs/etl_2026-10-12_14-28-43.log,None
3,2026-10-12 13:58:57,S25 FE,SUCCESS,8.57,LOADED,LOADED,LOADED,logs/etl_2026-10-12_13-58-57.log,None


## 2. Visão geral dos dados

In [4]:
listings.head(5)

,item_id,site_id,product_id,seller_id,accepts_mercadopago,price,currency_id,condition,listing_type_id,logistic_type,warranty,user_product_id,total_sales,price_converted,converted_currency,is_outlier,has_warranty,job_run
0,MLA2556093906,MLA,MLA60827731,2378399002,True,"1,055,916.47",ARS,new,gold_special,fulfillment,Sin garantía,NaN,NaN,696.06,USD,False,False,2026-10-12 14:28:43
1,MLA2117352013,MLA,MLA60827731,165489568,True,"1,399,884.00",ARS,new,gold_special,xd_drop_off,Garantía del vendedor: 6 meses,MLAU5281554196,NaN,922.80,USD,False,True,2026-10-12 14:28:43
2,MLA1774915413,MLA,MLA60827731,3224029707,True,"1,415,000.00",ARS,new,gold_special,xd_drop_off,Garantía del vendedor: 3 meses,MLAU3934472253,NaN,932.77,USD,False,True,2026-10-12 14:28:43
3,MLA2593287362,MLA,MLA60827731,378757741,True,"1,416,000.00",ARS,new,gold_special,xd_drop_off,Garantía del vendedor: 10 días,MLAU3618619997,NaN,933.43,USD,False,True,2026-10-12 14:28:43
4,MLA2147471291,MLA,MLA60827731,246960746,True,"1,580,000.00",ARS,new,gold_special,xd_drop_off,Garantía del vendedor: 6 meses,MLAU5341492379,NaN,"1,041.54",USD,False,True,2026-10-12 14:28:43


## 3. Pergunta 1 - Há algum vendedor com múltiplas publicações? Com quantas?

In [5]:
df1 = (
    listings
    .merge(sellers[['seller_id', 'nickname']], on='seller_id', how='left')
    .groupby(['seller_id', 'nickname'], dropna=False)
    .size()
    .reset_index(name='total_listings')
    .query('total_listings > 1')
    .sort_values('total_listings', ascending=False)
    .reset_index(drop=True)
)

total_sellers = listings['seller_id'].nunique()
print(f'{len(df1)} de {total_sellers} vendedores têm mais de uma publicação '
      f'(máximo: {df1["total_listings"].max() if len(df1) else 0}).')
df1.head(5)

36 de 65 vendedores têm mais de uma publicação (máximo: 7).


,seller_id,nickname,total_listings
0,2378399002,UNIVERSOELECTRONICOARF,7
1,2970851114,DETODITO_STORE_AR,7
2,246608163,TECHNOSHOPAR,7
3,165489568,FEDERICO BAGGINI,6
4,455337201,OMNIKA,6


## 4. Pergunta 2 - Média de vendas por vendedor
A API **não informa** a quantidade vendida por publicação (`total_sales` vem vazio), então a média calculada a partir dela é zero.

In [6]:
print(f'Listings com total_sales informado: {listings["total_sales"].notna().sum()} de {len(listings)}')
print(f'Média geral de vendas por vendedor (via listings): {listings["total_sales"].notna().mean():.2f}')

Listings com total_sales informado: 0 de 158
Média geral de vendas por vendedor (via listings): 0.00


## 5. Pergunta 3 - Qual é o preço médio em dólares?

In [7]:
avg_price_usd = listings[listings['is_outlier'].eq(False)]['price_converted'].mean()
print(f'Preço médio (sem outliers): US$ {avg_price_usd:,.2f}')

Preço médio (sem outliers): US$ 984.45


## 6. Pergunta 4 - Porcentagem de artigos com garantia

In [8]:
with_warranty = int(listings['has_warranty'].eq(True).sum())
no_warranty = int(listings['has_warranty'].eq(False).sum())
warranty_coverage = with_warranty / (with_warranty + no_warranty) * 100

print(f'{warranty_coverage:.2f}% das publicações têm algum tipo de garantia.')

79.75% das publicações têm algum tipo de garantia.


## 7. Pergunta 5 - Métodos de envio oferecidos

In [9]:
df5 = (
    listings['logistic_type']
    .value_counts(dropna=False)
    .rename('listings')
    .reset_index()
)
df5['percentage'] = (df5['listings'] / df5['listings'].sum() * 100).round(2)
df5

,logistic_type,listings,percentage
0,xd_drop_off,89,56.33
1,drop_off,34,21.52
2,fulfillment,21,13.29
3,cross_docking,14,8.86
